# Data Cleaning and Preparation

In this notebook, I prepare the dataset for my project **"What Makes a Rock Song a Hit?"**.  
The steps include:
- loading the full Spotify tracks dataset,
- filtering the rock-related genres,
- removing duplicate song-artist pairs,
- creating target and derived variables,
- and saving the cleaned dataset for later analysis.

In [31]:
import pandas as pd

df = pd.read_csv("../data/raw/dataset.csv")
print(df.shape)
df.head()

(114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [32]:
df["track_genre"].value_counts().sort_index()

track_genre
acoustic       1000
afrobeat       1000
alt-rock       1000
alternative    1000
ambient        1000
               ... 
techno         1000
trance         1000
trip-hop       1000
turkish        1000
world-music    1000
Name: count, Length: 114, dtype: int64

In [33]:
rock_df = df[df["track_genre"].str.contains("rock", case=False, na=False)].copy()

print("Rock-only dataset shape:", rock_df.shape)
print(rock_df["track_genre"].value_counts())

Rock-only dataset shape: (8000, 21)
track_genre
alt-rock       1000
hard-rock      1000
j-rock         1000
psych-rock     1000
punk-rock      1000
rock-n-roll    1000
rock           1000
rockabilly     1000
Name: count, dtype: int64


In [34]:
print("Exact duplicate rows:", rock_df.duplicated().sum())
print("Duplicate song+artist rows:", rock_df.duplicated(subset=["track_name", "artists"]).sum())

rock_df.isnull().sum().sort_values(ascending=False)

Exact duplicate rows: 0
Duplicate song+artist rows: 2123


Unnamed: 0          0
track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [35]:
rock_df = rock_df.drop(columns=["Unnamed: 0"])

## Duplicate Handling

Many songs appear multiple times in the dataset under the same track and artist combination.  
To avoid counting the same song more than once, I keep only one row per `track_name` + `artists` pair.  
When duplicates exist, I keep the version with the highest popularity score to ensure better analysis.

In [36]:
rock_df = (
    rock_df.sort_values("popularity", ascending=False)
           .drop_duplicates(subset=["track_name", "artists"])
           .copy()
)

print("Cleaned rock dataset shape:", rock_df.shape)
print(rock_df["track_genre"].value_counts())

Cleaned rock dataset shape: (5877, 20)
track_genre
j-rock         880
psych-rock     835
punk-rock      816
rock-n-roll    753
hard-rock      752
alt-rock       691
rockabilly     691
rock           459
Name: count, dtype: int64


## Feature Engineering

To support later analysis, I create:
- a binary `is_hit` variable based on the top 25% of popularity scores,
- `track_name_length`,
- `artist_count`,
- and `duration_min`.

In [37]:
hit_threshold = rock_df["popularity"].quantile(0.75)
rock_df["is_hit"] = (rock_df["popularity"] >= hit_threshold).astype(int)

rock_df["track_name_length"] = rock_df["track_name"].str.len()
rock_df["artist_count"] = rock_df["artists"].str.split(";").str.len()
rock_df["duration_min"] = rock_df["duration_ms"] / 60000

print("Hit threshold:", hit_threshold)
print(rock_df["is_hit"].value_counts())

Hit threshold: 57.0
is_hit
0    4349
1    1528
Name: count, dtype: int64


In [38]:
print(rock_df.shape)
print(rock_df.columns.tolist())
rock_df.head()

(5877, 24)
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'is_hit', 'track_name_length', 'artist_count', 'duration_min']


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,instrumentalness,liveness,valence,tempo,time_signature,track_genre,is_hit,track_name_length,artist_count,duration_min
91003,4h9wh7iOZ0GGn8QVp4RAOB,OneRepublic,I Ain’t Worried (Music From The Motion Picture...,I Ain't Worried,96,148485,False,0.704,0.797,0,...,0.000745,0.0546,0.825,139.994,4,rock,1,15,1,2.474750
91002,2QjOHCTQ1Jl3zawyYOpxh6,The Neighbourhood,I Love You.,Sweater Weather,93,240400,False,0.612,0.807,10,...,0.017700,0.1010,0.398,124.053,4,rock,1,15,1,4.006667
91001,5XeFesFbtLpXzIVDNQP22n,Arctic Monkeys,AM,I Wanna Be Yours,92,183956,False,0.464,0.417,0,...,0.022000,0.0974,0.479,67.528,4,rock,1,16,1,3.065933
91009,75FEaRjZTKLhTrFGsfMUXR,Kate Bush,Hounds Of Love,Running Up That Hill (A Deal With God),90,298933,False,0.629,0.547,10,...,0.003140,0.0604,0.197,108.375,4,rock,1,38,1,4.982217
91000,7DbdUf8aHSYoliSjO6LZv6,Beach Weather,Chit Chat,"Sex, Drugs, Etc.",90,196784,False,0.572,0.839,4,...,0.009760,0.5220,0.465,143.969,4,rock,1,16,1,3.279733


In [39]:
rock_df.to_csv("../data/clean/rock_dataset_cleaned.csv", index=False)

## Summary

The original dataset contained 114,000 songs across many genres.  
After filtering to rock-related genres, I obtained 8,000 rows across 8 rock subgenres.  
After removing duplicate song-artist pairs and keeping the most popular version of each duplicate, the final cleaned dataset contains 5,877 songs.

This cleaned dataset will be used for exploratory data analysis, hypothesis testing, and later machine learning tasks.